In [1]:
import os

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

!pwd
os.chdir("../RecSys_Course_AT_PoliMi")
!pwd

#!python run_compile_all_cython.py

/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/m2_kfold
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/RecSys_Course_AT_PoliMi


In [2]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from scipy.sparse import csr_matrix
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
#from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
#from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender

#from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

from Recommenders.hybrid.m2_model import IntegratedHierarchicalHybridRecommender
from Recommenders.hybrid.LinearHybridRecommender import GeneralizedLinearCoupleHybridRecommender


Tensorflow is not available


In [3]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

# Training knn costum similarity

In [4]:
SLIM_params = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

In [5]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))


In [6]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [7]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [8]:
prefitted_folds = []

for i in range(5):
    print(f"Fitting fold {i+1}/5...")
    
    # Creazione URM Train (Combined) e Test per questo fold
    URM_train = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
    URM_test = URM_parts[i]
    
    # Fit SLIM
    recommender_SLIM = SLIMElasticNetRecommender(URM_train)
    recommender_SLIM.fit(**SLIM_params)
    
    # Prepariamo l'evaluator per questo fold specifico
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])
    
    # Salviamo tutto in un dizionario per questo fold
    fold_data = {
        "URM_train": URM_train,
        "slim": recommender_SLIM,
        "evaluator": evaluator_test
    }
    
    prefitted_folds.append(fold_data)

print("Pre-training completato.")

Fitting fold 1/5...
SLIMElasticNetRecommender: Processed 4910 (70.5%) in 5.00 min. Items per second: 16.36
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.15 min. Items per second: 16.25
EvaluatorHoldout: Ignoring 34 ( 0.1%) Users that have less than 1 test interactions
Fitting fold 2/5...
SLIMElasticNetRecommender: Processed 4939 (70.9%) in 5.00 min. Items per second: 16.46
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.04 min. Items per second: 16.49
EvaluatorHoldout: Ignoring 31 ( 0.1%) Users that have less than 1 test interactions
Fitting fold 3/5...
SLIMElasticNetRecommender: Processed 4949 (71.0%) in 5.00 min. Items per second: 16.49
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.04 min. Items per second: 16.50
EvaluatorHoldout: Ignoring 36 ( 0.1%) Users that have less than 1 test interactions
Fitting fold 4/5...
SLIMElasticNetRecommender: Processed 4954 (71.1%) in 5.00 min. Items per second: 16.51
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 7.

In [ ]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
def objective_function_funksvd(optuna_trial):

                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        
        #Cambiare il modello qui sotto, insieme al range e ai parametri         
        recommender_instance = SLIM_BPR_Python(URM_combined)
        
        recommender_instance.fit(
                        topK = optuna_trial.suggest_int("topK", 1, 50),
                         learning_rate = optuna_trial.suggest_float("learning_rate", 1e-2, 9e-2, log=True),
                         lambda_i = optuna_trial.suggest_float("lambda_i", 1e-5, 5e-4, log=True),
                         lambda_j = optuna_trial.suggest_float("lambda_j", 1e-5, 2e-4, log=True),                         
                         # epochs = optuna_trial.suggest_int("epochs", 10, 50) 
                         )

        
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        #print("prova = ", result["MAP"].values[0])
        #print(result)
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    # Qui ripristinate le epochs
    # epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    # optuna_trial.set_user_attr("epochs", epochs) 

    
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


def objective_function_funksvd(optuna_trial):

    start_time = time.time()
    scores = []
    for fold_data in prefitted_folds:
        
        # Recuperiamo i modelli e i dati dal dizionario
        URM_train = fold_data["URM_train"]
        recommender_SLIM = fold_data["slim"]
        evaluator_test = fold_data["evaluator"]
        
        
        recommender = IntegratedHierarchicalHybridRecommender(
            URM_train, 
            recommender_SLIM, 
        )
        
        """alpha=optuna_trial.suggest_float("alpha", 0.60, 0.80)
        beta=optuna_trial.suggest_float("beta", 0.80, 1)"""

        topK = optuna_trial.suggest_int("topK", 600, 650)
        l1_ratio = optuna_trial.suggest_float("l1_ratio", 0.09, 0.1)
        alpha = optuna_trial.suggest_float("alpha", 0, 0.005)

        recommender.fit(topK, l1_ratio, alpha)
        
        
        result, _ = evaluator_test.evaluateRecommender(recommender)
        #print("prova = ", result["MAP"].values[0])
        #print(i)
        #print(result["RECALL"])
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [10]:
import optuna

optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 5)

[I 2025-12-27 17:59:13,137] A new study created in memory with name: no-name-fd9bad0c-204f-4b73-9313-d4374a5de999
[W 2025-12-27 17:59:13,153] Trial 0 failed with parameters: {} because of the following error: TypeError("IntegratedHierarchicalHybridRecommender.__init__() missing 2 required positional arguments: 'rec_sim_2' and 'rec_lin_3'").
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/wp/jydg89697jzcwnllv2_yz6d40000gn/T/ipykernel_28048/3386809055.py", line 31, in objective_function_funksvd
    recommender = IntegratedHierarchicalHybridRecommender(
        URM_train,
        recommender_SLIM,
    )
TypeError: IntegratedHierarchicalHybridRecommender.__init__() missing 2 required positional arguments: 'rec_sim_2' and 'rec_lin_3'
[W 2025-12-27 17:59:13,155] Trial 0 failed with value None.


TypeError: IntegratedHierarchicalHybridRecommender.__init__() missing 2 required positional arguments: 'rec_sim_2' and 'rec_lin_3'

# Da qui inizia il training su URM_all

In [ ]:
print()

In [ ]:
"""best_alpha_test = 0.7074505665346519
best_beta_test = 0.9199376036548086"""
#Trial 8 finished with value: 0.2909141858548001 and parameters: {'alpha': 0.7074505665346519, 'beta': 0.9199376036548086}

In [11]:
recommender_SLIM_f = SLIMElasticNetRecommender(URM_all)
recommender_SLIM_f.fit(**SLIM_params)

SLIMElasticNetRecommender: Processed 3418 (49.0%) in 5.00 min. Items per second: 11.39
SLIMElasticNetRecommender: Processed 6938 (99.6%) in 10.00 min. Items per second: 11.56
SLIMElasticNetRecommender: Processed 6969 (100.0%) in 10.05 min. Items per second: 11.55


In [ ]:
# Train the final model 
"""
new_similarity = (1 - best_alpha_test) * csr_matrix(recommender_rp3_f.W_sparse) + best_alpha_test * recommender_SLIM_f.W_sparse 
hybridrecommender_object_f = ItemKNNCustomSimilarityRecommender(URM_all)
hybridrecommender_object_f.fit(new_similarity)"""

In [ ]:
"""als_recommender_f = FeatureCombinedImplicitALSRecommender(URM_all)
als_recommender_f.fit(**IALS_params)"""

In [ ]:
"""recommenders = [hybridrecommender_object_f, als_recommender_f]

linear_comb_rec_f = GeneralizedLinearCoupleHybridRecommender(URM_all, recommenders)
linear_comb_rec_f.fit(best_beta_test)"""

In [12]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = recommender_SLIM_f.recommend(user_id, cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)
df_recommendations.to_csv("recommendations_slim.csv", index=False)

end_time = time.time()

       user_id                                          item_list
0            0  2530 2392 6411 5532 4486 828 3049 2168 2399 27...
1            1  6810 5630 4936 3460 2509 5223 2628 4483 2295 8...
2            2  4264 2557 5640 5323 5390 5369 1539 2529 5723 1...
3            3  4161 3713 336 6329 1458 6739 2873 729 6846 265...
4            4  4373 3512 2808 4835 2167 1638 5369 5356 4855 2...
...        ...                                                ...
27090    27090  2835 3325 4200 4231 336 4533 3316 4351 479 337...
27091    27091  3607 6096 2167 3441 5326 6562 554 2645 531 585...
27092    27092  2907 3516 3142 4936 1339 3907 1503 4709 1983 2...
27093    27093  4071 2985 1196 1373 1940 4989 2023 2855 2556 2...
27094    27094  596 1257 2252 6070 5723 4709 287 4809 6064 640...

[27095 rows x 2 columns]
